In [ ]:
# 1) Tuer les processus orphelins
!pkill -f streamlit
!pkill -f cloudflared

In [1]:
## CODE POUR LE MODELE BART



import subprocess, threading, time, re, os, json, hashlib

# 📦 Installation silencieuse des dépendances
subprocess.run("pip install -q streamlit PyPDF2 python-docx sentence-transformers faiss-cpu transformers torch evaluate sacrebleu rouge_score --extra-index-url https://download.pytorch.org/whl/cpu", shell=True)

# 🔧 Téléchargement de l'exécutable cloudflared
subprocess.run("wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", shell=True)
subprocess.run("chmod +x cloudflared", shell=True)

# 💻 Code Streamlit encapsulé dans une string
streamlit_code = r'''
import streamlit as st
import PyPDF2, docx, os, json, hashlib, re, time
import faiss, numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from evaluate import load

# 📌 Paramètres
EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
SUM_MODEL_NAME = "facebook/bart-large-cnn"
EMB_DIM = 384
FAISS_INDEX_PATH = "faiss_index.idx"
METADATA_PATH = "metadata.json"
TARGET_TOKENS = 350

# 🧠 Chargement des modèles avec cache
@st.cache_resource
def load_models():
    emb = SentenceTransformer(EMB_MODEL_NAME)
    summ = pipeline("summarization", model=SUM_MODEL_NAME)
    return emb, summ

# 📄 Extraction du texte selon le type de document
def extract_text(file):
    ext = file.name.split(".")[-1].lower()
    if ext == "pdf":
        return "".join(p.extract_text() or "" for p in PyPDF2.PdfReader(file).pages)
    elif ext == "docx":
        return "\n".join(p.text for p in docx.Document(file).paragraphs)
    elif ext == "txt":
        return file.read().decode("utf-8", errors="ignore")
    return ""

# ✂️ Segmentation et nettoyage du texte
def segment_and_clean(text):
    sentences = text.replace("\n", " ").split(".")
    segments, cur = [], ""
    for s in sentences:
        cur += s.strip() + ". "
        if len(cur.split()) >= TARGET_TOKENS:
            segments.append(cur.strip())
            cur = ""
    if cur: segments.append(cur.strip())
    return [re.sub(r"\s+", " ", s) for s in segments if s.strip()]

# 🔄 Chargement ou création de l'index FAISS
def build_or_load_faiss(emb):
    if os.path.exists(FAISS_INDEX_PATH):
        index = faiss.read_index(FAISS_INDEX_PATH)
        meta = json.load(open(METADATA_PATH, encoding="utf-8"))
    else:
        index = faiss.IndexFlatIP(EMB_DIM)
        meta = []
    return index, meta

# 💾 Sauvegarde de l'index et des métadonnées
def save_faiss(index, meta):
    faiss.write_index(index, FAISS_INDEX_PATH)
    json.dump(meta, open(METADATA_PATH, "w", encoding="utf-8"), ensure_ascii=False)

# ➕ Ajout de documents et segmentation
def add_documents(files, emb, index, meta):
    new_files = []
    for f in files:
        h = hashlib.md5(f.getbuffer()).hexdigest()
        if any(m["hash"] == h for m in meta): continue
        text = extract_text(f)
        segs = segment_and_clean(text)
        for i, seg in enumerate(segs):
            index.add(np.array(emb.encode([seg], normalize_embeddings=True), dtype="float32"))
            meta.append({"file": f.name, "segment_id": i, "text": seg, "hash": h})
        new_files.append(f.name)
    if new_files:
        save_faiss(index, meta)
    return new_files

# 🔍 Recherche de segments pertinents
def search(query, emb, index, meta, top_k=5):
    if index.ntotal == 0: return []
    q = emb.encode([query], normalize_embeddings=True)
    D, I = index.search(np.array(q, dtype="float32"), top_k)
    return [meta[i] for i in I[0] if i < len(meta)]

# 📝 Génération du résumé
def summarize(texts, summ):
    return summ(" ".join(texts)[:1500], max_length=130, min_length=30, do_sample=False)[0]["summary_text"]

# 📊 Fonction d'évaluation avec mise en cache intelligente
bleu_metric = load("bleu")
rouge_metric = load("rouge")

def cached_evaluation(summary, reference):
    key = hashlib.md5((summary + reference).encode("utf-8")).hexdigest()
    if not hasattr(st.session_state, "eval_cache"):
        st.session_state.eval_cache = {}
    if key in st.session_state.eval_cache:
        return st.session_state.eval_cache[key]
    b = bleu_metric.compute(predictions=[summary], references=[reference])["bleu"]
    r = rouge_metric.compute(predictions=[summary], references=[reference])
    st.session_state.eval_cache[key] = (b, r)
    return b, r

# 🎨 Interface Streamlit
st.set_page_config(page_title="🔍 RAG Optimisé", layout="wide")
st.title("📄 RAG Optimisé (Upload, Recherche, Résumé)")

emb, summ = load_models()
index, meta = build_or_load_faiss(emb)

# 📁 Upload
st.header("📁 Ajout de documents")
uploaded = st.file_uploader("Ajoutez vos documents", type=["pdf", "docx", "txt"], accept_multiple_files=True)
if uploaded:
    with st.spinner("🔄 Indexation automatique..."):
        added = add_documents(uploaded, emb, index, meta)
    if added:
        st.success(f"✅ Fichiers indexés : {', '.join(added)}")
    else:
        st.info("📎 Tous les fichiers étaient déjà indexés")

# 🔎 Recherche
st.header("🔍 Recherche et Résumé")
query = st.text_input("Posez une question")

if st.button("Chercher"):
    if not query.strip():
        st.warning("⚠️ Entrez une requête")
    else:
        with st.spinner("🔍 Traitement en cours..."):
            res = search(query, emb, index, meta)
            st.session_state.results = res
            valid_segments = [r["text"] for r in res if r.get("text", "").strip()]
            if valid_segments:
                summary = summarize(valid_segments, summ)
                st.session_state.summary = summary
            else:
                st.session_state.summary = ""

# 📑 Affichage des segments + résumé
if st.session_state.get("results"):
    st.subheader("📑 Segments pertinents")
    for r in st.session_state.results:
        with st.expander(f"{r['file']} §{r['segment_id']}"):
            st.write(r["text"])

if st.session_state.get("summary", "").strip():
    st.subheader("📝 Résumé généré")
    st.write(st.session_state.summary)

# 📊 Évaluation
st.header("📊 Évaluation du résumé")
if st.session_state.get("summary", "").strip():
    ref = st.text_area(
        "📄 Collez ou tapez votre résumé de référence",
        placeholder="Tapez ici votre résumé de référence...",
    )

    if ref.strip():
        # Message fixe pendant le calcul
        status = st.info("🔍 Calcul des scores BLEU / ROUGE…")

        # Calcul (avec fausse barre pour « ralentir » visuellement)
        bar = st.progress(0)
        for pct in range(0, 101, 20):
            time.sleep(0.15)
            bar.progress(pct)
        b, r = cached_evaluation(st.session_state.summary, ref)

        # Nettoyage progressif
        status.empty()
        bar.empty()

        # Affichage des scores dans trois rectangles colorés
        st.markdown(
            f"""
            <div style="display: flex; justify-content: space-around; gap: 1rem; margin-top: 1rem;">
                <div style="flex: 1; background:#3498db; color:white; padding:1em; border-radius:8px; text-align:center;">
                    <strong>🔵 BLEU</strong><br>{b:.3f}
                </div>
                <div style="flex: 1; background:#e74c3c; color:white; padding:1em; border-radius:8px; text-align:center;">
                    <strong>🔴 ROUGE-1</strong><br>{r['rouge1']:.3f}
                </div>
                <div style="flex: 1; background:#c0392b; color:white; padding:1em; border-radius:8px; text-align:center;">
                    <strong>🔴 ROUGE-L</strong><br>{r['rougeL']:.3f}
                </div>
            </div>
            """,
            unsafe_allow_html=True,
        )
else:
    st.info("Effectuez une recherche pour générer un résumé.")
'''

# 💾 Écriture du script Streamlit dans app.py
with open("app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_code)

# 🚀 Lancement Streamlit
subprocess.run("pkill -f streamlit || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)

subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port=8501",
    "--server.address=0.0.0.0",
    "--server.headless=true"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 🌐 Tunnel Cloudflare
url_holder = [None]
def tunnel():
    cmd = ["./cloudflared", "tunnel", "--url", "http://localhost:8501"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in iter(proc.stdout.readline, ""):
        match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
        if match:
            url_holder[0] = match.group(0)
            break

threading.Thread(target=tunnel, daemon=True).start()

print("⏳ Démarrage du tunnel...")
for _ in range(60):
    if url_holder[0]:
        print("🔗 URL publique :", url_holder[0])
        break
    time.sleep(1)
else:
    print("❌ Échec du tunnel Cloudflare")

KeyboardInterrupt: 

In [ ]:
# run this in your notebook / shell
!sed -i -e '1i from pathlib import Path' -e 's/\[key\}/key/' /content/app.py

In [2]:
## CODE POUR LE MODELE CHOISI T5-SMALL


import subprocess, threading, time, re, os, json, hashlib, textwrap, shutil
from pathlib import Path   # ✅ Ajout manquant

# ------------------------------------------------------------------
# 1️⃣  Installation silencieuse des dépendances
# ------------------------------------------------------------------
subprocess.run(
    "pip install -q streamlit PyPDF2 python-docx sentence-transformers "
    "faiss-cpu transformers torch evaluate sacrebleu rouge_score "
    "--extra-index-url https://download.pytorch.org/whl/cpu",
    shell=True,
)

# 2️⃣  Téléchargement de l’exécutable cloudflared
subprocess.run(
    "wget -q -O cloudflared "
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    shell=True,
)
subprocess.run("chmod +x cloudflared", shell=True)

# ------------------------------------------------------------------
# 3️⃣  Code Streamlit complet (string brute → app.py)
# ------------------------------------------------------------------
streamlit_code = r'''
import streamlit as st
import PyPDF2, docx, os, json, hashlib, re, time, shutil
import faiss, numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from evaluate import load
import textwrap
from pathlib import Path   # ✅ Ajout manquant

# ------------------------------------------------------------------
# 📌 Paramètres utilisateur
# ------------------------------------------------------------------
EMB_MODEL_NAME   = "sentence-transformers/all-MiniLM-L6-v2"
SUM_MODEL_NAME   = "t5-small"
SUM_MAX_INPUT    = 1500
SUM_MAX_LENGTH   = 130
SUM_MIN_LENGTH   = 30
EMB_DIM          = 384
FAISS_INDEX_PATH = "faiss_index.idx"
METADATA_PATH    = "metadata.json"
TARGET_TOKENS    = 350
CACHE_DIR        = ".cache_eval"

# ------------------------------------------------------------------
# 🧠 Chargement des modèles
# ------------------------------------------------------------------
@st.cache_resource
def load_models():
    emb = SentenceTransformer(EMB_MODEL_NAME)
    summ = pipeline(
        "summarization",
        model=SUM_MODEL_NAME,
        tokenizer=SUM_MODEL_NAME,
        device="cpu"
    )
    return emb, summ

# ------------------------------------------------------------------
# 📄 Extraction texte
# ------------------------------------------------------------------
def extract_text(file):
    ext = file.name.split(".")[-1].lower()
    if ext == "pdf":
        return "".join(p.extract_text() or "" for p in PyPDF2.PdfReader(file).pages)
    elif ext == "docx":
        return "\n".join(p.text for p in docx.Document(file).paragraphs)
    elif ext == "txt":
        return file.read().decode("utf-8", errors="ignore")
    return ""

# ------------------------------------------------------------------
# ✂️ Segmentation + nettoyage
# ------------------------------------------------------------------
def segment_and_clean(text):
    sentences = text.replace("\n", " ").split(".")
    segments, cur = [], ""
    for s in sentences:
        cur += s.strip() + ". "
        if len(cur.split()) >= TARGET_TOKENS:
            segments.append(cur.strip())
            cur = ""
    if cur:
        segments.append(cur.strip())
    return [re.sub(r"\s+", " ", s) for s in segments if s.strip()]

# ------------------------------------------------------------------
# 🔄 Chargement / création index FAISS
# ------------------------------------------------------------------
def build_or_load_faiss(emb):
    if os.path.exists(FAISS_INDEX_PATH):
        index = faiss.read_index(FAISS_INDEX_PATH)
        meta  = json.load(open(METADATA_PATH, encoding="utf-8"))
    else:
        index = faiss.IndexFlatIP(EMB_DIM)
        meta  = []
    return index, meta

# ------------------------------------------------------------------
# 💾 Sauvegarde index + métadonnées
# ------------------------------------------------------------------
def save_faiss(index, meta):
    faiss.write_index(index, FAISS_INDEX_PATH)
    json.dump(meta, open(METADATA_PATH, "w", encoding="utf-8"), ensure_ascii=False)

# ------------------------------------------------------------------
# ➕ Ajout documents + reset cache
# ------------------------------------------------------------------
def add_documents(files, emb, index, meta):
    new_files = []
    for f in files:
        h = hashlib.md5(f.getbuffer()).hexdigest()
        if any(m["hash"] == h for m in meta):
            continue
        text = extract_text(f)
        segs = segment_and_clean(text)
        for i, seg in enumerate(segs):
            index.add(np.array(emb.encode([seg], normalize_embeddings=True), dtype="float32"))
            meta.append({"file": f.name, "segment_id": i, "text": seg, "hash": h})
        new_files.append(f.name)
    if new_files:
        save_faiss(index, meta)
        _clear_eval_cache()
    return new_files

# ------------------------------------------------------------------
# 🔍 Recherche
# ------------------------------------------------------------------
def search(query, emb, index, meta, top_k=5):
    if index.ntotal == 0:
        return []
    q = emb.encode([query], normalize_embeddings=True)
    D, I = index.search(np.array(q, dtype="float32"), top_k)
    return [meta[i] for i in I[0] if i < len(meta)]

# ------------------------------------------------------------------
# 📝 Résumé T5
# ------------------------------------------------------------------
def summarize(texts, summ):
    context = " ".join(texts)[:SUM_MAX_INPUT]
    result = summ(
        "summarize: " + context,
        max_length=SUM_MAX_LENGTH,
        min_length=SUM_MIN_LENGTH,
        do_sample=False
    )
    return result[0]["summary_text"]

# ------------------------------------------------------------------
# 📊 Cache BLEU / ROUGE
# ------------------------------------------------------------------
bleu_metric  = load("bleu")
rouge_metric = load("rouge")

def _eval_cache_path(summary, reference):
    key = hashlib.md5((summary + reference).encode()).hexdigest()
    return Path(CACHE_DIR) / f"{key}.json"

def _clear_eval_cache():
    if os.path.isdir(CACHE_DIR):
        shutil.rmtree(CACHE_DIR)
    os.makedirs(CACHE_DIR, exist_ok=True)

def cached_evaluation(summary, reference):
    path = _eval_cache_path(summary, reference)
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    b = bleu_metric.compute(predictions=[summary], references=[reference])["bleu"]
    r = rouge_metric.compute(predictions=[summary], references=[reference])
    data = {"bleu": b, "rouge1": r["rouge1"], "rougeL": r["rougeL"]}
    path.write_text(json.dumps(data), encoding="utf-8")
    return data

# ------------------------------------------------------------------
# 🎨 Interface Streamlit
# ------------------------------------------------------------------
st.set_page_config(page_title="🔍 RAG T5 Optimisé", layout="wide")
st.title("🧪 Génération & Évaluation de Résumé")

emb, summ = load_models()
index, meta = build_or_load_faiss(emb)

# ------------------------------------------------
# 📁 Upload de documents
# ------------------------------------------------
st.header("📁 Ajout de documents")
uploaded = st.file_uploader(
    "Ajoutez vos documents",
    type=["pdf", "docx", "txt"],
    accept_multiple_files=True
)
if uploaded:
    with st.spinner("🔄 Indexation automatique..."):
        added = add_documents(uploaded, emb, index, meta)
    if added:
        st.success(f"✅ Fichiers indexés : {', '.join(added)}")
    else:
        st.info("📎 Tous les fichiers étaient déjà indexés")

# ------------------------------------------------
# 🔎 Recherche (reset des résultats à chaque clic)
# ------------------------------------------------
st.header("🔍 Recherche")
query = st.text_input("Posez une question")
if st.button("Chercher"):
    if not query.strip():
        st.warning("⚠️ Entrez une requête")
    else:
        with st.spinner("🔍 Traitement en cours.."):
            # ✅ Nettoyage de l’historique
            for key in ("results", "summary", "ref_input", "show_bleu_button"):
                st.session_state.pop(key, None)

            res = search(query, emb, index, meta)
            st.session_state.results = res
            valid = [r["text"] for r in res if r.get("text", "").strip()]
            st.session_state.summary = summarize(valid, summ) if valid else ""
            _clear_eval_cache()

# ------------------------------------------------
# 📑 Affichage des segments
# ------------------------------------------------
if st.session_state.get("results"):
    st.subheader("📑 Segments pertinents")
    for r in st.session_state.results:
        with st.expander(f"{r['file']} - segment {r['segment_id']}"):
            st.write(r["text"])

# ------------------------------------------------
# 📝 Résumé généré
# ------------------------------------------------
if st.session_state.get("summary"):
    st.subheader("📝 Résumé T5")
    st.write(st.session_state.summary)

# ------------------------------------------------
# 📊 Bouton d’évaluation (toujours visible après résumé)
# ------------------------------------------------
if st.session_state.get("summary"):
    st.header("📊 Évaluation du résumé")
    ref = st.text_area(
        "📄 Résumé de référence",
        placeholder="Collez ici votre résumé de référence...",
        key="ref_input"
    )
    # Bouton placé AVANT la vérification du contenu
    if st.button("Calculer les scores BLEU / ROUGE"):
        if ref.strip():
            with st.spinner("🔍 Calcul des scores BLEU / ROUGE…"):
                bar = st.progress(0)
                for pct in range(0, 101, 25):
                    time.sleep(0.15)
                    bar.progress(pct)
                scores = cached_evaluation(st.session_state.summary, ref)
                bar.empty()

            html = textwrap.dedent(f"""
                <div style="display:flex;justify-content:space-around;gap:1rem;margin-top:1rem;">
                  <div style="flex:1;background:#3498db;color:white;padding:1em;border-radius:8px;text-align:center;">
                    BLEU&nbsp;&nbsp;{scores['bleu']:.3f}
                  </div>
                  <div style="flex:1;background:#e74c3c;color:white;padding:1em;border-radius:8px;text-align:center;">
                    ROUGE-1&nbsp;&nbsp;{scores['rouge1']:.3f}
                  </div>
                  <div style="flex:1;background:#c0392b;color:white;padding:1em;border-radius:8px;text-align:center;">
                    ROUGE-L&nbsp;&nbsp;{scores['rougeL']:.3f}
                  </div>
                </div>
            """).strip()
            st.markdown(html, unsafe_allow_html=True)
        else:
            st.warning("⚠️ Veuillez coller un résumé de référence pour calculer les scores.")
else:
    st.info("ℹ️ Effectuez une recherche pour générer un résumé avant de calculer les scores.")
'''

# ------------------------------------------------------------------
# 4️⃣  Écriture de app.py
# ------------------------------------------------------------------
Path("app.py").write_text(streamlit_code, encoding="utf-8")

# ------------------------------------------------------------------
# 5️⃣  Lancement Streamlit + tunnel Cloudflare
# ------------------------------------------------------------------
subprocess.run("pkill -f streamlit || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)

subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port=8501",
    "--server.address=0.0.0.0",
    "--server.headless=true"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

url_holder = [None]

def tunnel():
    cmd = ["./cloudflared", "tunnel", "--url", "http://localhost:8501"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in iter(proc.stdout.readline, ""):
        match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
        if match:
            url_holder[0] = match.group(0)
            break

threading.Thread(target=tunnel, daemon=True).start()

print("⏳ Démarrage du tunnel...")
for _ in range(60):
    if url_holder[0]:
        print("🔗 URL publique :", url_holder[0])
        break
    time.sleep(1)
else:
    print("❌ Échec du tunnel Cloudflare")

⏳ Démarrage du tunnel...
🔗 URL publique : https://clan-lies-colleges-airport.trycloudflare.com
